In [1]:
import pandas as pd 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
import torchvision
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('./dataset/gk_qna_dataset.csv')

In [3]:
df['question'][0]

'What is the capital of "France"?'

In [4]:
import re
def tokenize_text(text):
    text = text.lower()
    text = re.sub( r"[\"']","",text)
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    text = re.sub(r"\s+"," ", text).strip()
    token = text.split()
    return token

In [5]:
tokenize_text(df['question'][0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [6]:
vocab = {'<UNK>':0}

def build_vocab (row):
    tokenize_question = tokenize_text(row['question'])
    tokenize_answer = tokenize_text(row['answer'])
    marge_all = tokenize_question + tokenize_answer
    for token in marge_all:
        if token not in vocab:
            vocab[token]=len(token)
df.apply(build_vocab,axis=1)

0      None
1      None
2      None
3      None
4      None
       ... 
166    None
167    None
168    None
169    None
170    None
Length: 171, dtype: object

In [7]:
vocab

{'<UNK>': 0,
 'what': 4,
 'is': 2,
 'the': 3,
 'capital': 7,
 'of': 2,
 'france': 6,
 'paris': 5,
 'germany': 7,
 'berlin': 6,
 'italy': 5,
 'rome': 4,
 'spain': 5,
 'madrid': 6,
 'japan': 5,
 'tokyo': 5,
 'canada': 6,
 'ottawa': 6,
 'brazil': 6,
 'brasilia': 8,
 'australia': 9,
 'canberra': 8,
 'india': 5,
 'new': 3,
 'delhi': 5,
 'china': 5,
 'beijing': 7,
 'russia': 6,
 'moscow': 6,
 'united': 6,
 'states': 6,
 'washington': 10,
 'dc': 2,
 'mexico': 6,
 'city': 4,
 'egypt': 5,
 'cairo': 5,
 'turkey': 6,
 'ankara': 6,
 'argentina': 9,
 'buenos': 6,
 'aires': 5,
 'south': 5,
 'korea': 5,
 'seoul': 5,
 'indonesia': 9,
 'jakarta': 7,
 'pakistan': 8,
 'islamabad': 9,
 'bangladesh': 10,
 'dhaka': 5,
 'nepal': 5,
 'kathmandu': 9,
 'sri': 3,
 'lanka': 5,
 'colombo': 7,
 'thailand': 8,
 'bangkok': 7,
 'malaysia': 8,
 'kuala': 5,
 'lumpur': 6,
 'vietnam': 7,
 'hanoi': 5,
 'uae': 3,
 'abu': 3,
 'dhabi': 5,
 'iran': 4,
 'tehran': 6,
 'iraq': 4,
 'baghdad': 7,
 'saudi': 5,
 'arabia': 6,
 'riyadh

In [8]:
print(len(vocab),vocab['what'])

243 4


In [ ]:
def text_to_indices (text,vocab):
    indexed_text = []
    for token in tokenize_text(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else :
            indexed_text.append(vocab['<UNK>'])
    return indexed_text
text_to_indices("the is Phitron",vocab)
vocal = { "<UNK>":0}


    

[3, 2, 0]

In [10]:
index = 0

text_to_indices(df.iloc[index]['question'],vocab)

[4, 2, 3, 7, 2, 6]

In [11]:
class QADataset():
    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab
    def __len__(self):
        return self.df.shape[0]
    def __getitem__(self,index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)
        return torch.tensor(numerical_question),torch.tensor(numerical_answer)
dataset = QADataset(df,vocab)

In [12]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)
for i in dataloader:
    print(i)

[tensor([[2, 3, 4, 5, 3, 2, 6, 6]]), tensor([[6, 7]])]
[tensor([[3, 4, 5, 4, 5, 6, 2, 3, 7, 2, 5]]), tensor([[4, 5]])]
[tensor([[2, 3, 4, 5, 4, 6, 3]]), tensor([[7]])]
[tensor([[ 3, 10, 10]]), tensor([[9, 7]])]
[tensor([[5, 8, 4, 2, 3, 4, 8, 2, 6]]), tensor([[10]])]
[tensor([[4, 2, 3, 7, 2, 6]]), tensor([[5]])]
[tensor([[1, 3, 9, 4, 2, 4, 4, 3]]), tensor([[4, 8]])]
[tensor([[5, 6, 2, 5, 2, 3, 3, 6]]), tensor([[4]])]
[tensor([[4, 4, 3, 5, 3]]), tensor([[ 8, 10,  4]])]
[tensor([[5, 6, 2, 3, 7, 2, 5]]), tensor([[4, 5]])]
[tensor([[1, 3, 9, 5, 5, 5, 5]]), tensor([[5]])]
[tensor([[ 3,  4,  5,  4,  3,  9,  3,  6,  2, 10]]), tensor([[6, 8]])]
[tensor([[1, 3, 9, 3, 8, 3, 5, 4]]), tensor([[6, 6]])]
[tensor([[4, 4, 3, 4]]), tensor([[11, 11,  9]])]
[tensor([[2, 3, 4, 4, 2, 3, 4, 8, 2, 6]]), tensor([[10]])]
[tensor([[1, 3, 9, 4, 4, 3, 4]]), tensor([[11, 11,  9]])]
[tensor([[2, 3, 4, 4, 2, 3, 7, 5]]), tensor([[7, 5]])]
[tensor([[2, 3, 4, 3, 8, 3, 5, 4]]), tensor([[6, 6]])]
[tensor([[3, 3, 4, 2, 3, 

In [ ]:
criterion = nn.CrossEntropyLoss()